# Who is a neobank actually built for?

"Neobank" gets used as one category, but a third of them are not chasing the general
public at all. They are built for one group — freelancers, migrants, teenagers, SMBs,
one faith community — and that choice shows up in how they are regulated and when they
were founded.

This notebook maps that segmentation across all 368 tracked neobanks using the
`audience` field, which is populated for every row. Nothing here needs cleaning or
imputation, which is why it is a good place to start.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Kaggle mounts inputs in two different places depending on how the dataset was
# attached: /kaggle/input/<slug>/ from the UI, /kaggle/input/datasets/<owner>/<slug>/
# when declared through the API. Searching for the file covers both, and keeps this
# working if you fork the notebook and attach the data yourself.
def find_entities():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for hit in sorted(kaggle_input.rglob("entities.csv")):
            return hit
    for local in (Path("entities.csv"), Path("../../.staging/entities.csv")):
        if local.exists():
            return local
    return None


src = find_entities()
if src is None:
    raise FileNotFoundError(
        "entities.csv not found. On Kaggle, add the neobankbeat/neobanks dataset via "
        "'Add Input'. Locally, download it from "
        "https://www.kaggle.com/datasets/neobankbeat/neobanks"
    )
df = pd.read_csv(src)

plt.rcParams.update({"figure.figsize": (9, 5), "axes.spines.top": False, "axes.spines.right": False})
print(f"{len(df)} neobanks · {df.shape[1]} columns")
df.head(3)

## 1. How many serve a niche rather than everyone?

`audience` is `general` for a mass-market product, and a named segment otherwise.

In [ ]:
aud = df["audience"].fillna("unknown")
niche_mask = aud.ne("general")

print(f"general audience : {(~niche_mask).sum():>3}")
print(f"built for a niche: {niche_mask.sum():>3}  ({100 * niche_mask.mean():.0f}% of the directory)")
print(f"distinct niches  : {aud[niche_mask].nunique()}")

segments = aud[niche_mask].value_counts()
ax = segments.sort_values().plot.barh(color="#0f766e")
ax.set(title="Neobanks built for a specific audience", xlabel="neobanks", ylabel="")
plt.tight_layout()
plt.show()

segments.to_frame("neobanks")

## 2. The long tail is the interesting part

Business banking is the obvious niche and it dominates. Below it sits a genuine long
tail — several segments with only one or two products worldwide, which is a rough map
of who is currently underserved.

In [ ]:
tail = segments[segments <= 3]
print(f"{tail.size} segments have 3 or fewer products worldwide:\n")
for name, n in tail.items():
    who = ", ".join(df.loc[aud.eq(name), "name"].head(4))
    print(f"  {name:26} {n}  — {who}")

## 3. Niche products cluster in different regions

Segments are not evenly spread. Underbanked-focused products track emerging markets;
freelancer and SMB products track Europe and the US.

In [ ]:
top_segments = segments.head(6).index
sub = df[aud.isin(top_segments)].copy()
sub["audience"] = aud[sub.index]

cross = pd.crosstab(sub["audience"], sub["region"])
cross = cross.loc[top_segments, cross.sum().sort_values(ascending=False).index]

ax = cross.plot.bar(stacked=True, colormap="viridis", figsize=(10, 5))
ax.set(title="Where niche neobanks are based, by segment", xlabel="", ylabel="neobanks")
ax.legend(title="region", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

cross

## 4. Do niche products get licensed less often?

This is the question worth asking of the segmentation: a product serving a vulnerable
group — migrants, the underbanked — matters more if it holds its own licence, because
that determines what happens to deposits if it fails.

In [ ]:
reg = df["regulation_type"].fillna("Undisclosed")
licensed = reg.eq("Licensed bank")

summary = pd.DataFrame({
    "neobanks": [(~niche_mask).sum(), niche_mask.sum()],
    "hold own bank licence": [licensed[~niche_mask].sum(), licensed[niche_mask].sum()],
}, index=["general", "niche"])
summary["% licensed"] = (100 * summary["hold own bank licence"] / summary["neobanks"]).round(1)

print(summary.to_string())
print()

order = reg.value_counts().head(5).index
share = pd.crosstab(niche_mask.map({False: "general", True: "niche"}), reg, normalize="index")[order] * 100
ax = share.plot.bar(figsize=(10, 5), colormap="cividis")
ax.set(title="How each group is authorised (% within group)", xlabel="", ylabel="% of group")
ax.legend(title="regulation_type", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

share.round(1)

## 5. Niche banking is the newer wave

Plot founding years and the segmentation reads as a timeline: mass-market challengers
first, then products aimed at groups the first wave did not serve well.

In [ ]:
years = df.dropna(subset=["founded"]).copy()
years["group"] = niche_mask[years.index].map({False: "general", True: "niche"})
years = years[years["founded"].between(2005, 2026)]

pivot = years.pivot_table(index="founded", columns="group", values="name", aggfunc="count").fillna(0)
ax = pivot.plot.area(alpha=0.75, color=["#94a3b8", "#0f766e"], figsize=(10, 5))
ax.set(title="Neobanks founded per year, general vs niche", xlabel="year founded", ylabel="launches")
plt.tight_layout()
plt.show()

median = years.groupby("group")["founded"].median()
print(f"Median founding year — general: {median.get('general', float('nan')):.0f} · niche: {median.get('niche', float('nan')):.0f}")

## 6. Pull the list for one audience

Change `SEGMENT` to any value from the table in section 1.

In [ ]:
SEGMENT = "freelancers & creators"  # try "underbanked", "SMB & startups", "gen z & students"

cols = ["name", "hq", "founded", "regulation_type", "custody", "card_type", "website"]
picks = df.loc[aud.eq(SEGMENT), cols].sort_values("founded")

print(f"{len(picks)} neobanks for: {SEGMENT}")
picks.reset_index(drop=True)

## Caveats, and how to cite this

Three things to know before quoting any number above.

**Coverage is deliberate.** Only live, consumer-facing products are tracked. Defunct
neobanks and pure BaaS/infrastructure providers are excluded, so this cannot be used
for survival analysis — the denominator is "what exists now", not "what was ever launched".

**Empty is not zero.** Unverified fields are left blank rather than guessed, so a missing
value means "not confirmed from a primary source", not "does not have it". Treat every
count here as a floor.

**Self-disclosed figures are not audited.** Where user counts, funding or volume appear,
they are what companies chose to announce, on the metric they chose to announce it on.

Data: [neobankbeat/neobanks](https://www.kaggle.com/datasets/neobankbeat/neobanks) ·
methodology and field dictionary: [neobankbeat.com/data/](https://www.neobankbeat.com/data/)

> neobankbeat (2026). *Open directory of neobanks worldwide.* https://www.neobankbeat.com/ (MIT).